In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load Dataset (30 features, 569 instances)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# 2. Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Scale Features (Crucial for SVM, KNN, and Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler and test data for Streamlit deployment
os.makedirs("model", exist_ok=True)
joblib.dump(scaler, "model/scaler.pkl")

test_df = pd.DataFrame(X_test)
test_df['target'] = y_test.values
test_df.to_csv("test_data.csv", index=False)

print("Dataset loaded, scaled, and test_data.csv saved successfully!")
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

Dataset loaded, scaled, and test_data.csv saved successfully!
Training samples: 455, Test samples: 114


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# Initialize models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    # Train model on scaled training data
    model.fit(X_train_scaled, y_train)

    # Save trained model artifact
    filename = f"model/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(model, filename)

    # Predict on test data
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

results_df = pd.DataFrame(results)
print("\n--- Model Performance Comparison Table ---")
display(results_df)


--- Model Performance Comparison Table ---


/home/s2026ak05257/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9825,0.9825,0.9825,0.9623
1,Decision Tree,0.9123,0.9157,0.9161,0.9123,0.9130,0.8174
2,KNN,0.9561,0.9788,0.9561,0.9561,0.9560,0.9054
3,Naive Bayes,0.9298,0.9868,0.9298,0.9298,0.9298,0.8492
4,Support Vector Machine,0.9825,0.9950,0.9825,0.9825,0.9825,0.9623


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

st.title("M.Tech ML Assignment 2 - Classification Model Evaluation")
st.write("Deployed via Streamlit Community Cloud for BITS Pilani WILP (AIML)")

# Model selection dropdown
model_name = st.selectbox(
    "Select Classification Model",
    ["Logistic Regression", "Decision Tree", "KNN", "Naive Bayes", "Support Vector Machine"]
)

model_files = {
    "Logistic Regression": "model/logistic_regression.pkl",
    "Decision Tree": "model/decision_tree.pkl",
    "KNN": "model/knn.pkl",
    "Naive Bayes": "model/naive_bayes.pkl",
    "Support Vector Machine": "model/support_vector_machine.pkl"
}

# Load artifacts
model = joblib.load(model_files[model_name])
scaler = joblib.load("model/scaler.pkl")

# File uploader for test CSV
uploaded_file = st.file_uploader("Upload your test dataset CSV (e.g., test_data.csv)", type=["csv"])

if uploaded_file is not None:
    test_df = pd.read_csv(uploaded_file)
    st.write("### Test Data Preview", test_df.head())

    X_test = test_df.iloc[:, :-1]
    y_test = test_df.iloc[:, -1]

    # Scale test features using saved scaler
    X_test_scaled = scaler.transform(X_test)

    # Predictions
    y_pred = model.predict(X_test_scaled)
    try:
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    except Exception:
        auc = 0.0

    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    st.write(f"### Evaluation Metrics for {model_name}")
    col1, col2, col3 = st.columns(3)
    col1.metric("Accuracy", f"{acc:.4f}")
    col1.metric("AUC Score", f"{auc:.4f}")
    col2.metric("Precision", f"{prec:.4f}")
    col2.metric("Recall", f"{rec:.4f}")
    col3.metric("F1 Score", f"{f1:.4f}")
    col3.metric("MCC Score", f"{mcc:.4f}")

    # Confusion Matrix
    st.write("### Confusion Matrix")
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    st.pyplot(fig)

    st.write("### Classification Report")
    report = classification_report(y_test, y_pred, output_dict=True)
    st.dataframe(pd.DataFrame(report).transpose())

Overwriting app.py


In [4]:
%%writefile requirements.txt
streamlit==1.67.0
scikit-learn==1.9.1
numpy==2.5.4
pandas==2.2.3
matplotlib==3.10.1
seaborn==0.13.2
joblib==1.4.2

Overwriting requirements.txt
